# DAMO-YOLO 自訂資料集訓練 → RK3588 RKNN 部署

本專案已直接內建 **PyTorch 2.6+ 相容性修復**、**AMP 混合精度加速** 以及 **RKNN 專屬工具（tools_custom）**。

這本 notebook 會依序完成：

1. 準備環境，從你自己的 GitHub 取得已包含修復與工具的 DAMO-YOLO 原始碼
2. 從 **Google Drive**（COCO JSON 或 YOLO txt）或 **Roboflow** 取得資料集，統一轉成 DAMO-YOLO 需要的 COCO 格式
3. 用 COCO 預訓練權重做遷移訓練（T / S / M），內建 AMP 加速（T4 上快 1.5～2 倍）
4. 對 test 圖片推論，畫出 bbox 檢查結果
5. 匯出成 **RKNN 友善的 ONNX**：6 個原始輸出，DFL、decode、NMS 全部留給 CPU
6. 用 rknn-toolkit2 做 **INT8 量化**並轉成 `.rknn`，接著在模擬器上比較 ONNX 和 RKNN 的 mAP
7. 打包 `.rknn`、標籤檔、後處理程式和板端推論程式到 Google Drive

**使用前：** 選單「執行階段 → 變更執行階段類型 → T4 GPU」（或更好的 GPU）。

**Colab 斷線時：** 訓練的 checkpoint 都存在 Google Drive。重新連線後，執行 ⚙️ 參數設定，勾選 `RESUME`，再依序執行第 1、2、3 節即可續訓。


## ⚙️ 參數設定

In [ ]:
# @title ⚙️ 參數設定 { display-mode: "form" }
# @markdown ### 專案
PROJECT_NAME = "my_damoyolo"  # @param {type:"string"}
MODEL_SIZE = "S"  # @param ["T", "S", "M"]
# @markdown 所有輸出（checkpoint、ONNX、RKNN）都存到這個 Google Drive 資料夾
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/damoyolo_runs"  # @param {type:"string"}

# @markdown ---
# @markdown ### 資料集來源
DATA_SOURCE = "google_drive"  # @param ["google_drive", "roboflow"]
# @markdown **google_drive**：`.zip` 檔或資料夾都可以；格式可以是 COCO JSON 或 YOLO txt（`auto` 會自動判斷）
DRIVE_DATASET_PATH = "/content/drive/MyDrive/dataset.zip"  # @param {type:"string"}
DATASET_FORMAT = "auto"  # @param ["auto", "coco", "yolo"]
# @markdown 資料集沒有 val split 時，從 train 隨機切出的比例
VAL_RATIO = 0.15  # @param {type:"number"}
# @markdown **roboflow**：API key 請放在 Colab 左側「🔑 Secrets」，名稱設為 `ROBOFLOW_API_KEY`，並開啟 notebook 存取權
ROBOFLOW_WORKSPACE = ""  # @param {type:"string"}
ROBOFLOW_PROJECT = ""  # @param {type:"string"}
ROBOFLOW_VERSION = 1  # @param {type:"integer"}

# @markdown ---
# @markdown ### 訓練
# @markdown `IMG_SIZE` 必須是 32 的倍數；RK3588 上 640 是常見的平衡點，要更快可用 512 或 416
IMG_SIZE = 640  # @param {type:"integer"}
# @markdown 從 COCO 預訓練權重 finetune，數千張圖通常 40～60 epoch 就夠了；請以 3.3 的 mAP 曲線判斷
EPOCHS = 50  # @param {type:"integer"}
# @markdown T4 建議：T / S 用 16～32，M 用 8～16；遇到 CUDA out of memory 就調小
BATCH_SIZE = 16  # @param {type:"integer"}
EVAL_INTERVAL = 5  # @param {type:"integer"}
# @markdown checkpoint 間隔越短，續訓或縮短排程時可以接續的點越多（每個約 100 MB 以上）
CKPT_INTERVAL = 5  # @param {type:"integer"}
# @markdown 混合精度訓練（AMP），T4 上約可快 1.5～2 倍；如果 loss 出現 nan，取消勾選後續訓
USE_AMP = True  # @param {type:"boolean"}
# @markdown 勾選後會從最新的 checkpoint 續訓（Colab 斷線時使用）
RESUME = False  # @param {type:"boolean"}
# @markdown 留空會自動下載官方 COCO 預訓練權重；也可以填自己的 `.pth` 路徑
PRETRAINED_PATH = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### RKNN 轉換
RKNN_PLATFORM = "rk3588"  # @param ["rk3588", "rk3576", "rk3568", "rk3566", "rk3562"]
# @markdown 量化校正用的圖片張數（從 train 隨機抽取）
QUANT_IMAGES = 200  # @param {type:"integer"}
QUANT_ALGO = "normal"  # @param ["normal", "mmse", "kl_divergence"]
# @markdown 在模擬器上比較 ONNX 和 RKNN mAP 用的 val 圖片張數（模擬器較慢）
RKNN_EVAL_IMAGES = 200  # @param {type:"integer"}

# @markdown ---
# @markdown ### DAMO-YOLO 原始碼
# @markdown 請填入你上傳至 GitHub 的專案網址（已整合修復與 tools_custom）
REPO_SOURCE = "github"  # @param ["github", "drive_zip"]
GITHUB_REPO_URL = "https://github.com/<YOUR_USERNAME>/DAMO-YOLO.git"  # @param {type:"string"}
REPO_ZIP_PATH = "/content/drive/MyDrive/DAMO-YOLO-master.zip"  # @param {type:"string"}

import re
PROJECT_NAME = re.sub(r'\W', '_', PROJECT_NAME.strip()) or 'my_damoyolo'
REPO = '/content/DAMO-YOLO'
DATA_OUT = '/content/datasets/custom'      # 轉換後的資料集（Colab 本機磁碟）
RAW_DIR = '/content/raw_dataset'           # 原始資料集解壓位置
RUN_DIR = f'{DRIVE_OUTPUT_ROOT}/{PROJECT_NAME}'
EXP_NAME = f'{PROJECT_NAME}_{MODEL_SIZE}'
WORK_DIR = f'{RUN_DIR}/workdirs'
CKPT_DIR = f'{WORK_DIR}/{EXP_NAME}'
EXPORT_DIR = f'{RUN_DIR}/export'
print('專案輸出位置：', RUN_DIR)

## 1. 環境準備
### 1.1 掛載 Google Drive、確認 GPU

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(RUN_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1.2 取得 DAMO-YOLO 原始碼並安裝套件

In [ ]:
import glob, shutil, sys
if not os.path.isdir(f'{REPO}/damo'):
    if REPO_SOURCE == 'github':
        repo_url = GITHUB_REPO_URL.strip()
        if not repo_url or '<YOUR_USERNAME>' in repo_url:
            raise ValueError('請在「⚙️ 參數設定」的 GITHUB_REPO_URL 填入您的 GitHub 專案網址！')
        !git clone --depth 1 "{repo_url}" {REPO}
    else:
        tmp = '/content/_repo_zip'
        shutil.rmtree(tmp, ignore_errors=True)
        !unzip -q "{REPO_ZIP_PATH}" -d {tmp}
        found = glob.glob(f'{tmp}/**/damo/__init__.py', recursive=True)
        assert found, '這個 zip 裡找不到 damo/ 資料夾，請確認是 DAMO-YOLO 的原始碼'
        shutil.move(os.path.dirname(os.path.dirname(found[0])), REPO)

!pip install -q easydict loguru tabulate thop omegaconf timm scikit-image pycocotools onnx onnxruntime gdown pyyaml
# onnxsim 只是用來簡化 ONNX，沒有預編譯版本時略過，不影響轉換
!pip install -q --only-binary=:all: onnxsim || echo "onnxsim 略過"
if DATA_SOURCE == 'roboflow':
    !pip install -q roboflow

os.environ['PYTHONPATH'] = REPO
# DAMO 的設定檔用相對路徑讀取 backbone 結構（./damo/...），所以工作目錄必須是 repo 根目錄
os.chdir(REPO)
# 資料夾要先存在再加入 sys.path：不存在的路徑會被 Python 快取成「不可匯入」，
# 之後即使寫入檔案也 import 不到
for p in (REPO, f'{REPO}/tools_custom'):
    if p not in sys.path:
        sys.path.insert(0, p)
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

### 1.3 內建相容性與工具說明
本專案已在原始碼中直接整合以下優化，**不再需要動態打補丁或手動寫入檔案**：

| 類別 | 項目 | 說明 |
|---|---|---|
| **相容性修復** | PyTorch ≥ 2.6 | 修復 `torch.load(weights_only=True)` 導致 checkpoint 無法載入的問題 |
| | `torchrun` 支援 | 支援 `--local-rank` 與 `LOCAL_RANK` 環境變數 |
| | 自訂資料集註冊 | `paths_catalog.py` 內建 `custom_train_coco` 與 `custom_val_coco` |
| **訓練加速** | 混合精度 (AMP) | 卷積 FP16、損失維持 FP32，T4 上速度提升約 1.5～2 倍 |
| | cuDNN & TF32 | 固定輸入尺寸自動尋優 kernel，Ampere GPU 啟用 TF32 |
| | DDP 優化 | 關閉 `find_unused_parameters`，減少多餘計算圖遍歷 |
| **RKNN 專用工具** | `tools_custom/` | 內建資料集轉換、自動組態、RKNN-friendly ONNX 匯出、INT8 量化與板端推論 |


In [ ]:
# 驗證 tools_custom 工具程式與專案已就緒
import tools_custom
print('✅ 專案與 tools_custom 工具已就緒，可直接進行資料集準備與訓練！')


## 2. 資料集

### 2.1 下載並轉換資料集
支援的格式：

- **COCO JSON**
  - Roboflow 匯出格式：`train/`、`valid/`、`test/`，每個資料夾內有 `_annotations.coco.json` 和圖片
  - 官方 COCO 格式：`annotations/instances_train*.json` 加上 `train*/`、`val*/` 圖片資料夾
- **YOLO txt**
  - `images/{train,val}` + `labels/{train,val}`
  - Roboflow 的 `train/images`、`train/labels`
  - 圖片和 `.txt` 放在同一個資料夾
  - 類別名稱從 `data.yaml` 或 `classes.txt` 讀取
  - 分割（segmentation）格式的多邊形標註會自動轉成外框

處理規則：

- 沒有 val split 時，會從 train 隨機切出 `VAL_RATIO`
- 只有 test split 時，會把 test 當成 val
- 沒有任何標註框的類別（例如 Roboflow 自動加的上層類別）會自動移除

In [ ]:
import shutil, json, importlib, sys
TOOLS = f'{REPO}/tools_custom'
missing = [f for f in ('prepare_dataset.py', 'make_config.py', 'damo_rknn_utils.py', 'export_onnx.py',
                       'rknn_convert.py', 'board_infer.py') if not os.path.exists(f'{TOOLS}/{f}')]
assert not missing, f'找不到 {missing}，請確認專案 tools_custom 是否完整'
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)
sys.path_importer_cache.pop(TOOLS, None)   # 清掉資料夾建立前留下的匯入快取
importlib.invalidate_caches()
from prepare_dataset import prepare_dataset

shutil.rmtree(RAW_DIR, ignore_errors=True)
os.makedirs(RAW_DIR)
fmt = DATASET_FORMAT
if DATA_SOURCE == 'google_drive':
    assert os.path.exists(DRIVE_DATASET_PATH), f'找不到 {DRIVE_DATASET_PATH}'
    if os.path.isfile(DRIVE_DATASET_PATH) and DRIVE_DATASET_PATH.lower().endswith('.zip'):
        # 先複製到本機磁碟再解壓，比直接從 Drive 讀取快很多
        shutil.copy(DRIVE_DATASET_PATH, '/content/_dataset.zip')
        !unzip -q -o /content/_dataset.zip -d {RAW_DIR}
    elif os.path.isdir(DRIVE_DATASET_PATH):
        shutil.copytree(DRIVE_DATASET_PATH, RAW_DIR, dirs_exist_ok=True)
    else:
        raise ValueError('DRIVE_DATASET_PATH 必須是 .zip 檔或資料夾')
else:
    from roboflow import Roboflow
    try:
        from google.colab import userdata
        api_key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        from getpass import getpass
        api_key = getpass('沒有在 Secrets 找到 ROBOFLOW_API_KEY，請輸入 Roboflow API key：')
    rf = Roboflow(api_key=api_key)
    (rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
       .version(ROBOFLOW_VERSION).download('coco', location=RAW_DIR, overwrite=True))
    fmt = 'coco'

CLASS_NAMES = prepare_dataset(RAW_DIR, DATA_OUT, fmt=fmt, val_ratio=VAL_RATIO)
shutil.copy(f'{DATA_OUT}/classes.json', RUN_DIR)

### 2.2 檢查資料集

In [ ]:
import collections, random, json
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

for split in ('train', 'val'):
    d = json.load(open(f'{DATA_OUT}/annotations/{split}.json'))
    cnt = collections.Counter(a['category_id'] for a in d['annotations'])
    print(f'[{split}] {len(d["images"])} 張圖、{len(d["annotations"])} 個框')
    for i, n in enumerate(CLASS_NAMES):
        print(f'    {i:3d} {n:25s} {cnt.get(i + 1, 0)}')

d = json.load(open(f'{DATA_OUT}/annotations/train.json'))
anns = collections.defaultdict(list)
for a in d['annotations']:
    anns[a['image_id']].append(a)
samples = random.sample(d['images'], min(6, len(d['images'])))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax in axes.ravel():
    ax.axis('off')
for ax, im in zip(axes.ravel(), samples):
    ax.imshow(cv2.imread(f'{DATA_OUT}/train/{im["file_name"]}')[:, :, ::-1])
    for a in anns[im['id']]:
        x, y, w, h = a['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color='lime', lw=2))
        ax.text(x, y, CLASS_NAMES[a['category_id'] - 1], color='black', fontsize=9,
                bbox=dict(facecolor='lime', alpha=0.7, pad=1, lw=0))
    ax.set_title(im['file_name'][:40], fontsize=9)
plt.tight_layout(); plt.show()

## 3. 訓練

### 3.1 產生設定檔、下載預訓練權重
- 以官方的 T / S / M 設定為範本，自動填入類別數、類別名稱、輸入尺寸、epochs 和 batch 等設定
- 預訓練權重：README 裡阿里雲的連結已失效（404），這裡改用 README 中 Google Drive 的鏡像。backbone 和 neck 與目前的設定檔相容；分類 head 會因類別數不同重新初始化，這是遷移訓練的正常行為

In [ ]:
import glob, re, json
from make_config import make_config, TEMPLATES

if 'CLASS_NAMES' not in globals():
    CLASS_NAMES = json.load(open(f'{DATA_OUT}/classes.json'))['class_names']

_, gdrive_id, fname = TEMPLATES[MODEL_SIZE]
if PRETRAINED_PATH:
    pretrained = PRETRAINED_PATH
else:
    pretrained = f'{DRIVE_OUTPUT_ROOT}/pretrained/{fname}'   # 多個專案共用，只下載一次
    if not os.path.exists(pretrained) or os.path.getsize(pretrained) < 1_000_000:
        import gdown
        os.makedirs(os.path.dirname(pretrained), exist_ok=True)
        gdown.download(id=gdrive_id, output=pretrained, quiet=False)
assert os.path.getsize(pretrained) > 1_000_000, f'預訓練權重下載失敗：{pretrained}'

def epoch_ckpts():
    files = glob.glob(f'{CKPT_DIR}/epoch_*_ckpt.pth')
    return sorted(files, key=lambda p: int(re.search(r'epoch_(\d+)_ckpt', p).group(1)))

resume_path = None
if RESUME:
    if os.path.exists(f'{CKPT_DIR}/latest_ckpt.pth'):
        print('⚠️ 已經有 latest_ckpt.pth，代表訓練已完成；如要重新訓練請取消勾選 RESUME')
    elif epoch_ckpts():
        resume_path = epoch_ckpts()[-1]
        print('從這個 checkpoint 續訓：', resume_path)
    else:
        print('⚠️ 沒有找到 checkpoint，改為從頭訓練')

CONFIG_PATH = make_config(
    REPO, MODEL_SIZE, EXP_NAME, CLASS_NAMES, WORK_DIR, img_size=IMG_SIZE,
    epochs=EPOCHS, batch_size=BATCH_SIZE, eval_interval=EVAL_INTERVAL,
    ckpt_interval=CKPT_INTERVAL, finetune_path=pretrained, resume_path=resume_path,
    num_workers=max(2, os.cpu_count() or 2))   # AMP 之後 GPU 變快，資料讀取要跟上
shutil.copy(CONFIG_PATH, RUN_DIR)
from damo.config.base import parse_config
os.chdir(REPO)   # 設定檔內用相對路徑 ./damo/... 讀取 backbone 結構
REG_MAX = parse_config(CONFIG_PATH).model.head.reg_max
print('設定檔：', CONFIG_PATH)
print(open(CONFIG_PATH).read().split('# ---- custom dataset overrides (generated) ----')[1])

### 3.2 開始訓練
- 每 `EVAL_INTERVAL` 個 epoch 會在 val 上算一次 mAP；最後 `no_aug_epochs` 個 epoch 會關閉 mosaic 增強，並改成每個 epoch 都評估
- 每 `CKPT_INTERVAL` 個 epoch 存一次 checkpoint，訓練完成時另存 `latest_ckpt.pth`（EMA 權重）
- 輸出會同時附加到 Drive 上的 `train_log.txt`（原專案的 log 在續訓時會被覆寫，這份不會），3.3 會用它整理 mAP

In [ ]:
TRAIN_LOG = f'{RUN_DIR}/train_log.txt'
os.environ['DAMO_AMP'] = '1' if USE_AMP else '0'
os.environ['PYTHONUNBUFFERED'] = '1'
import datetime
with open(TRAIN_LOG, 'a', encoding='utf-8') as f:
    f.write(f'===== start {datetime.datetime.now():%Y-%m-%d %H:%M:%S} | AMP={USE_AMP} | {CONFIG_PATH} =====\n')
!cd {REPO} && PYTHONPATH={REPO} torchrun --nproc_per_node=1 --master_port=29501 tools/train.py -f "{CONFIG_PATH}" 2>&1 | tee -a "{TRAIN_LOG}"

### 3.3 查看 mAP 曲線、判斷是否提前結束
**訓練中也可以執行這格**（Colab 會排隊，等訓練那格結束才執行；要邊訓練邊看的話，請在 Drive 上直接打開 `train_log.txt`）。

判斷方式：
- 看 **mAP50-95**，不要看 loss。mosaic 等資料增強會讓 loss 跟真實精度脫鉤
- 學習率開始下降之後，**連續 3 次評估的提升都不到 0.5 個百分點**，代表進入平台期
- **不要直接中斷訓練**：最後 `no_aug_epochs` 個 epoch 會關閉 mosaic、固定在最低學習率，mAP 通常在這段明顯跳升。正確的做法是停止訓練後，把 `EPOCHS` 改小、勾選 `RESUME`，重新執行 3.1 和 3.2，排程會重新計算，並且一樣跑完最後的 no-aug 階段

In [ ]:
import re
import matplotlib.pyplot as plt
TRAIN_LOG = f'{RUN_DIR}/train_log.txt'
pat_ep = re.compile(r'epoch: (\d+)/(\d+), iter: \d+/\d+')
pat_ap = re.compile(r'Average Precision\s+\(AP\) @\[ IoU=(0\.50:0\.95|0\.50)\s+\| area=\s+all \| maxDets=100 \] = (-?[\d.]+)')
epoch, total, by_epoch = None, None, {}
for line in open(TRAIN_LOG, encoding='utf-8', errors='ignore'):
    m = pat_ep.search(line)
    if m:
        epoch, total = int(m.group(1)), int(m.group(2))
        continue
    m = pat_ap.search(line)
    if m and epoch is not None:
        key = 'mAP50-95' if m.group(1) == '0.50:0.95' else 'mAP50'
        by_epoch.setdefault(epoch, {})[key] = float(m.group(2))   # 續訓時以最新一次為準

rows = sorted((e, v) for e, v in by_epoch.items() if 'mAP50-95' in v)
if not rows:
    print('還沒有評估結果（每', EVAL_INTERVAL, '個 epoch 評估一次）')
else:
    print(f'{"epoch":>6} {"mAP50-95":>9} {"mAP50":>7}')
    for e, v in rows:
        print(f'{e:>6} {v["mAP50-95"]:>9.4f} {v.get("mAP50", float("nan")):>7.4f}')
    best_e, best_v = max(rows, key=lambda r: r[1]['mAP50-95'])
    print(f'\n目前最佳：epoch {best_e}，mAP50-95 = {best_v["mAP50-95"]:.4f}（排程總長 {total} epoch）')
    if len(rows) >= 4:
        before = max(v['mAP50-95'] for _, v in rows[:-3])
        recent = max(v['mAP50-95'] for _, v in rows[-3:])
        gain = recent - before
        if gain < 0.005:
            print(f'最近 3 次評估只提升 {gain * 100:+.2f} 個百分點，可能已進入平台期：'
                  '可以考慮把 EPOCHS 改小並勾選 RESUME，讓排程提早進入最後的 no-aug 階段。')
        else:
            print(f'最近 3 次評估提升 {gain * 100:+.2f} 個百分點，還在進步中。')
    xs = [e for e, _ in rows]
    plt.figure(figsize=(8, 4))
    plt.plot(xs, [v['mAP50-95'] for _, v in rows], 'o-', label='mAP50-95')
    plt.plot(xs, [v.get('mAP50', float('nan')) for _, v in rows], 's--', label='mAP50')
    plt.xlabel('epoch'); plt.ylabel('mAP'); plt.grid(alpha=0.3); plt.legend(); plt.show()

### 3.4 選擇最終權重、清理 checkpoint
DAMO-YOLO 不會自動挑出最佳權重，這裡優先使用訓練完成時存的 `latest_ckpt.pth`；如果訓練中斷，則改用 epoch 數最大的 checkpoint。

In [ ]:
# @title 選擇最終權重 { display-mode: "form" }
# @markdown 勾選後會刪除中間的 epoch checkpoint（每個約 100 MB 以上），只保留最終權重
DELETE_INTERMEDIATE = True  # @param {type:"boolean"}
import glob, re, torch
if os.path.exists(f'{CKPT_DIR}/latest_ckpt.pth'):
    FINAL_CKPT = f'{CKPT_DIR}/latest_ckpt.pth'
else:
    files = sorted(glob.glob(f'{CKPT_DIR}/epoch_*_ckpt.pth'),
                   key=lambda p: int(re.search(r'epoch_(\d+)_ckpt', p).group(1)))
    assert files, f'{CKPT_DIR} 裡沒有 checkpoint'
    FINAL_CKPT = files[-1]
print('最終權重：', FINAL_CKPT, '| epoch', torch.load(FINAL_CKPT, map_location='cpu', weights_only=False).get('epoch'))
if DELETE_INTERMEDIATE:
    for p in glob.glob(f'{CKPT_DIR}/epoch_*_ckpt.pth'):
        if p != FINAL_CKPT:
            os.remove(p); print('刪除', p)

## 4. 測試圖片推論（PyTorch）
用最終權重對 **test 資料夾的前 `NUM_TEST_IMAGES` 張圖片**推論，畫出 bbox 並顯示在下方；畫好的圖也會存到 Drive 的 `test_results/`。

- `TEST_DIR` 留空時，會自動在原始資料集裡找名為 `test` 的資料夾（Roboflow 的 `test/`、YOLO 的 `images/test` 都可以）；也可以填任何圖片資料夾的路徑
- 找不到 test 資料夾時，會改用 val 的圖片
- 推論使用跟 RK3588 上相同的前處理和 numpy 後處理（`damo_rknn_utils.py`），所以這裡的結果也可以當作部署前的檢查

In [ ]:
# @title 測試圖片推論 { display-mode: "form" }
# @markdown 留空：自動在原始資料集裡找 `test` 資料夾
TEST_DIR = "/content/raw_dataset/dataset/test"  # @param {type:"string"}
NUM_TEST_IMAGES = 10  # @param {type:"integer"}
TEST_CONF = 0.3  # @param {type:"number"}
import glob, json, math
import cv2, numpy as np, torch
import matplotlib.pyplot as plt
import damo_rknn_utils as U
from export_onnx import build_model

assert 'FINAL_CKPT' in globals(), '請先執行 3.4 選擇最終權重'
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def list_images(d):
    return sorted(p for p in glob.glob(f'{d}/**/*', recursive=True) if p.lower().endswith(IMG_EXTS))

def find_test_dir(root):
    cands = [dp for dp, _, _ in os.walk(root)
             if os.path.basename(dp).lower() in ('test', 'testing', 'test2017')]
    for d in sorted(cands, key=len):          # 最上層的 test 資料夾優先
        if list_images(d):
            return d
    return None

test_dir = TEST_DIR.strip() or find_test_dir(RAW_DIR)
if test_dir is None:
    test_dir = f'{DATA_OUT}/val'
    print('⚠️ 原始資料集裡沒有 test 資料夾，改用 val 圖片')
elif not TEST_DIR.strip() and json.load(open(f'{DATA_OUT}/classes.json')).get('val_source') == 'test':
    print('⚠️ 這個資料集沒有 val split，test 已經被當成 val 使用（訓練時用來評估，但沒有參與訓練）')
images = list_images(test_dir)[:NUM_TEST_IMAGES]
assert images, f'{test_dir} 裡沒有圖片'
print(f'測試資料夾：{test_dir}（取前 {len(images)} 張）')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _ = build_model(CONFIG_PATH, FINAL_CKPT, device=device)
out_dir = f'{RUN_DIR}/test_results'
os.makedirs(out_dir, exist_ok=True)

cols = 2
rows = math.ceil(len(images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 6 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis('off')
for ax, path in zip(axes, images):
    img = cv2.imread(path)
    h, w = img.shape[:2]
    x = torch.from_numpy(U.to_nchw(U.preprocess(img, IMG_SIZE))).to(device)
    with torch.no_grad():
        outs = [o.float().cpu().numpy() for o in model(x)]
    boxes, scores, cls_ids = U.postprocess(outs, IMG_SIZE, (w, h), conf_thr=TEST_CONF,
                                           iou_thr=0.45, reg_max=REG_MAX)
    vis = U.draw(img, boxes, scores, cls_ids, CLASS_NAMES)
    cv2.imwrite(f'{out_dir}/{os.path.basename(path)}', vis)
    counts = {}
    for c in cls_ids:
        counts[CLASS_NAMES[c]] = counts.get(CLASS_NAMES[c], 0) + 1
    print(f'{os.path.basename(path)[:50]:50s} {len(boxes):3d} 個物件 {counts}')
    ax.imshow(vis[:, :, ::-1])
    ax.set_title(f'{os.path.basename(path)[:45]}  |  {len(boxes)} objects', fontsize=10)
plt.tight_layout(); plt.show()
print('結果已存到：', out_dir)

## 5. 匯出 ONNX（RKNN 用）
這裡用的匯出程式是為 RKNN 量身打造的，跟官方 `converter.py` 有幾個差異：

- 模型裡不含後處理：沒有 concat、DFL softmax 和 NMS
- 6 個輸出依序為 `cls_8, reg_8, cls_16, reg_16, cls_32, reg_32`
  - `cls` 已經過 sigmoid，形狀是 K×H×W
  - `reg` 是 DFL logits，形狀是 68×H×W（4 個邊 × 17 個 bin）
- 輸入是 1×3×S×S 的 RGB，數值範圍 0～255，直接 resize（不用 letterbox），不做 mean / std 正規化

In [ ]:
os.makedirs(EXPORT_DIR, exist_ok=True)
ONNX_PATH = f'{EXPORT_DIR}/{EXP_NAME}_{IMG_SIZE}.onnx'
!cd {REPO} && PYTHONPATH={REPO} python tools_custom/export_onnx.py -f "{CONFIG_PATH}" -c "{FINAL_CKPT}" --img_size {IMG_SIZE} -o "{ONNX_PATH}"

### 5.1 驗證 ONNX 和後處理
用跟板端相同的 numpy 後處理（`damo_rknn_utils.py`）計算 val 的 mAP，數值應該接近 3.3 中訓練最後一次評估的結果。

In [ ]:
import random, cv2
import onnxruntime as ort
import damo_rknn_utils as U
import matplotlib.pyplot as plt

sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
def onnx_predict(img_bgr, conf=0.001, multi_label=True):
    h, w = img_bgr.shape[:2]
    outs = sess.run(None, {'images': U.to_nchw(U.preprocess(img_bgr, IMG_SIZE))})
    return U.postprocess(outs, IMG_SIZE, (w, h), conf_thr=conf, iou_thr=0.65,
                         reg_max=REG_MAX, multi_label=multi_label)

ONNX_MAP = U.coco_map(onnx_predict, f'{DATA_OUT}/annotations/val.json', f'{DATA_OUT}/val', max_images=500)
print(f'ONNX mAP50-95 {ONNX_MAP[0]:.4f} | mAP50 {ONNX_MAP[1]:.4f}')

val_imgs = sorted(os.listdir(f'{DATA_OUT}/val'))
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, fn in zip(axes, random.sample(val_imgs, min(3, len(val_imgs)))):
    img = cv2.imread(f'{DATA_OUT}/val/{fn}')
    b, s, c = onnx_predict(img, conf=0.3, multi_label=False)
    ax.imshow(U.draw(img, b, s, c, CLASS_NAMES)[:, :, ::-1]); ax.axis('off'); ax.set_title(fn[:40])
plt.tight_layout(); plt.show()

## 6. 轉換 RKNN（INT8 量化）

### 6.1 安裝 rknn-toolkit2（獨立環境）
rknn-toolkit2 會鎖定特定版本的 torch、onnx 和 protobuf，所以用 `uv` 建立一個 **Python 3.11 的獨立環境**，避免弄壞訓練環境，也不受 Colab 的 Python 版本影響。

In [ ]:
!pip install -q uv
!uv venv -q --python 3.11 /content/rknn_env
!uv pip install -q --python /content/rknn_env/bin/python rknn-toolkit2 pycocotools opencv-python-headless --extra-index-url https://download.pytorch.org/whl/cpu --index-strategy unsafe-best-match
!/content/rknn_env/bin/python -c "from rknn.api import RKNN; print('rknn-toolkit2 OK')"
!uv pip show --python /content/rknn_env/bin/python rknn-toolkit2 | head -2

### 6.2 量化並在模擬器上驗證
- 從 train 隨機抽 `QUANT_IMAGES` 張圖做量化校正（先 resize 到模型輸入尺寸）
- `mean_values=0`、`std_values=1`，輸入格式是 RGB uint8 NHWC，跟訓練時一致
- 驗證會輸出兩組數字：
  - ONNX 和 RKNN 模擬器各輸出的**餘弦相似度**（> 0.99 為正常）
  - 前 `RKNN_EVAL_IMAGES` 張 val 圖的 **mAP 比較**，用來看量化掉了多少精度

如果精度掉太多，可以依序嘗試：

1. `QUANT_ALGO` 改成 `mmse`（較慢，但通常較準）
2. 增加 `QUANT_IMAGES`
3. 勾選 `RUN_ANALYSIS` 查看逐層誤差，再用混合量化把誤差大的層保留成 FP16

In [ ]:
# @title 量化並轉換 { display-mode: "form" }
# @markdown 勾選後輸出逐層誤差分析到 export/snapshot（耗時較久）
RUN_ANALYSIS = False  # @param {type:"boolean"}
import random, cv2
import damo_rknn_utils as U

qdir = '/content/quant_images'
shutil.rmtree(qdir, ignore_errors=True); os.makedirs(qdir)
train_imgs = sorted(os.listdir(f'{DATA_OUT}/train'))
random.Random(0).shuffle(train_imgs)
lines = []
for fn in train_imgs[:QUANT_IMAGES]:
    img = cv2.imread(f'{DATA_OUT}/train/{fn}')
    if img is None:
        continue
    rgb = U.preprocess(img, IMG_SIZE)
    p = f'{qdir}/{len(lines):05d}.jpg'
    cv2.imwrite(p, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
    lines.append(p)
open(f'{qdir}/dataset.txt', 'w').write('\n'.join(lines))
print(f'量化校正圖片：{len(lines)} 張')

RKNN_PATH = f'{EXPORT_DIR}/{EXP_NAME}_{IMG_SIZE}_{RKNN_PLATFORM}.rknn'
cmd = (f'cd {REPO}/tools_custom && /content/rknn_env/bin/python rknn_convert.py'
       f' --onnx "{ONNX_PATH}" --out "{RKNN_PATH}" --dataset {qdir}/dataset.txt'
       f' --img_size {IMG_SIZE} --platform {RKNN_PLATFORM} --quant_algo {QUANT_ALGO} --reg_max {REG_MAX}'
       f' --val_json {DATA_OUT}/annotations/val.json --val_dir {DATA_OUT}/val'
       f' --eval_images {RKNN_EVAL_IMAGES} --report "{EXPORT_DIR}/rknn_report.json"'
       + (' --analysis' if RUN_ANALYSIS else ''))
!{cmd}

## 7. 打包輸出
`export/` 資料夾（在 Google Drive）會包含：

| 檔案 | 用途 |
|---|---|
| `*.rknn` | RK3588 用的模型 |
| `*.onnx` | FP32 ONNX，可在 PC 上驗證，或之後重新量化 |
| `*.pth` | 只含權重的 PyTorch 檔，可用於之後再 finetune |
| `model_info.json` | 輸入尺寸、類別、輸出順序等資訊 |
| `labels.txt` | 類別名稱，一行一個 |
| `damo_rknn_utils.py` | 前處理和後處理（numpy，可直接用在板端） |
| `board_infer.py` | 板端推論範例（rknn-toolkit-lite2） |
| `rknn_report.json` | 量化前後的精度比較 |

In [ ]:
# @title 打包 { display-mode: "form" }
DOWNLOAD_ZIP = False  # @param {type:"boolean"}
import json, torch
ck = torch.load(FINAL_CKPT, map_location='cpu', weights_only=False)
torch.save({'model': ck['model']}, f'{EXPORT_DIR}/{EXP_NAME}.pth')
info = {
    'model': os.path.basename(RKNN_PATH),
    'model_size': MODEL_SIZE,
    'img_size': IMG_SIZE,
    'reg_max': REG_MAX,
    'strides': [8, 16, 32],
    'num_classes': len(CLASS_NAMES),
    'class_names': CLASS_NAMES,
    'input': 'RGB uint8, 1xSxSx3 (NHWC) for RKNN / 1x3xSxS float 0-255 for ONNX; '
             'plain resize to SxS, no letterbox, no mean/std',
    'outputs': ['cls_8', 'reg_8', 'cls_16', 'reg_16', 'cls_32', 'reg_32'],
    'output_note': 'cls: 1xKxHxW sigmoid scores; reg: 1x(4*(reg_max+1))xHxW DFL logits for (l,t,r,b). '
                   'd = sum(softmax(bins) * [0..reg_max]) * stride; '
                   'box = (x*stride - l, y*stride - t, x*stride + r, y*stride + b), x/y = grid cell index',
}
json.dump(info, open(f'{EXPORT_DIR}/model_info.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
open(f'{EXPORT_DIR}/labels.txt', 'w', encoding='utf-8').write('\n'.join(CLASS_NAMES) + '\n')
for f in ('damo_rknn_utils.py', 'board_infer.py', 'rknn_convert.py'):
    shutil.copy(f'{REPO}/tools_custom/{f}', EXPORT_DIR)
print('輸出資料夾：', EXPORT_DIR)
for f in sorted(os.listdir(EXPORT_DIR)):
    p = f'{EXPORT_DIR}/{f}'
    if os.path.isfile(p):
        print(f'  {f:45s} {os.path.getsize(p) / 1e6:8.2f} MB')

if DOWNLOAD_ZIP:
    from google.colab import files
    z = shutil.make_archive(f'/content/{EXP_NAME}_export', 'zip', EXPORT_DIR)
    files.download(z)

## 8. 板端部署（RK3588）

**1. 在板子上安裝 rknn-toolkit-lite2**

```bash
pip3 install rknn-toolkit-lite2 opencv-python-headless numpy
```

板子上的 `librknnrt.so` 版本要 **≥ 轉換時使用的 rknn-toolkit2 版本**（版本在 6.1 的輸出中）。如果不符，到 [airockchip/rknn-toolkit2](https://github.com/airockchip/rknn-toolkit2) 的 `rknpu2/runtime` 取得對應版本，並更新 `/usr/lib/librknnrt.so`。

**2. 把 Drive 上的 `export/` 資料夾複製到板子，然後執行**

```bash
python3 board_infer.py --source test.jpg                      # 單張圖片，結果存到 result.jpg
python3 board_infer.py --source 0 --core 0                    # USB 攝影機
python3 board_infer.py --source video.mp4 --save out.mp4      # 影片
```

**3. 提升吞吐量**

RK3588 有 3 個 NPU 核心。單一小模型拆到多核心上跑效果有限，比較好的做法是開 3 個 process 或 thread，分別用 `--core 0/1/2`，輪流處理影格。

**4. 用 C++ 部署**

可以參考 [airockchip/rknn_model_zoo](https://github.com/airockchip/rknn_model_zoo) 的 C API 範例。後處理照著 `damo_rknn_utils.py` 的 `decode()` 和 `postprocess()` 改寫即可：

- 對 `reg` 的 17 個 bin 做 softmax，再乘上 0～16 的權重求加權和，得到距離，再乘上 stride
- 框的計算方式：`(x*stride - l, y*stride - t, x*stride + r, y*stride + b)`，其中 x、y 是格點座標
- 縮放回原圖尺寸，最後做 NMS